In [ ]:
from IPython.display import Image, display

import glob, sys

import numpy as np
import math
import hvplot.xarray

import matplotlib.pyplot as plt
import pandas as pd

from scipy.ndimage import gaussian_filter, distance_transform_edt
from skimage.restoration import denoise_tv_chambolle, inpaint_biharmonic


sys.path.append('../')
from config import CONFIG

sys.path.append('../../EMIT-Data-Resources/python/modules/')
from emit_tools import emit_xarray
help(emit_xarray)

In [ ]:
imfns = glob.glob(f"{CONFIG['plot_folder']}/Intermountain/*")
for i, fn in enumerate(imfns):
    print(f"{i}\t{fn}")

In [ ]:
display(Image(filename=imfns[5]))

In [ ]:
rad_fps = glob.glob(f"{CONFIG['data_folder']}/Martin_Lake/*L1B_RAD*")
for i, fn in enumerate(rad_fps):
    print(f"{i}\t{fn}")

In [ ]:
fp = rad_fps[20]
ds_geo = emit_xarray(fp, ortho=True)

In [ ]:
import numpy as np
import xarray as xr

def rgb_stretch(rgb_ds, qlo=2, qhi=98, gamma=2.2, white_background=False):
    da = rgb_ds["radiance"]  # or "reflectance" if you have it

    out = []
    for wl in da["wavelengths"].values:
        b = da.sel(wavelengths=wl)

        # mask nonpositive / invalid
        b = b.where(b > 0)

        lo = b.quantile(qlo/100.0)
        hi = b.quantile(qhi/100.0)

        b = (b - lo) / (hi - lo)
        b = b.clip(0, 1)

        # gamma correction (display gamma)
        b = b ** (1/gamma)

        if white_background:
            b = b.fillna(1.0)
        out.append(b)

    da_out = xr.concat(out, dim="wavelengths")
    da_out = da_out.assign_coords(wavelengths=da["wavelengths"])

    rgb_ds = rgb_ds.copy()
    rgb_ds["radiance"] = da_out
    return rgb_ds

rgb = ds_geo.sel(wavelengths=[700, 529, 470], method="nearest")
rgb = rgb_stretch(rgb, qlo=2, qhi=98, gamma=2.2, white_background=True)

map_rgb = rgb.hvplot.rgb(x="longitude", y="latitude", bands="wavelengths",
                         aspect="equal", frame_height=500)
map_rgb


In [ ]:
loc_name = 'Colstrip'

mean_stack = np.load(f"{CONFIG['results_folder']}/{CONFIG['tavg_subdir']}/{loc_name}/{loc_name}_mean_stack.npy")

fig, axs = plt.subplots(1, 3, figsize=(14,4), layout='constrained')

v = mean_stack[np.isfinite(mean_stack)]
lim = np.nanpercentile(np.abs(v), 99.5)   # tighter than 99.5
im = axs[0].imshow(mean_stack, cmap='RdBu_r', vmin=-lim, vmax=lim)
plt.colorbar(im, ax=axs[0])
axs[0].scatter(mean_stack.shape[1] // 2, mean_stack.shape[0] // 2, marker='x', s=50, c='green')
axs[0].set_title('Raw dSCD')

im = axs[1].imshow(gaussian_filter(mean_stack, 2), cmap="RdBu_r", vmin=-lim, vmax=lim)
plt.colorbar(im, ax=axs[1])
axs[1].scatter(mean_stack.shape[1] // 2, mean_stack.shape[0] // 2, marker='x', s=50, c='green')
axs[1].set_title('Gaussian filter')


mask = np.isfinite(mean_stack)
# nearest-neighbor fill
_, idx = distance_transform_edt(~mask, return_indices=True)
filled = mean_stack[tuple(idx)]   # copies nearest valid value into each NaN pixel
tv = denoise_tv_chambolle(filled, weight=0.2)
tv[~mask] = np.nan


im = axs[2].imshow(tv, cmap='RdBu_r', origin='upper', vmin=-0.001, vmax=0.001)
plt.colorbar(im, ax=axs[2])
axs[2].scatter(mean_stack.shape[1] // 2, mean_stack.shape[0] // 2, marker='x', s=50, c='green')
axs[2].set_title('TV filter')

fig.suptitle(f"{loc_name} Power Plant Time Averaged Retrieval", fontsize=20)

In [ ]:
tavg_fps = glob.glob(f"{CONFIG['plot_folder']}/*/*Time_Average*")
for i, fn in enumerate(tavg_fps):
    print(f"{i}\t{fn}")

[1,1,0,1,1,.5,1,0.5,1,1,1,.5,0,1,1,1,1,1]

In [ ]:
display(Image(filename=tavg_fps[17]))

In [ ]:
csv_savefn = f"{CONFIG['data_folder']}/metadata.csv"
data = pd.read_csv(csv_savefn)
s_dat = data[data['PLUME']==True]
f_dat = data[data['PLUME']==False]
print(len(data))

sspd = (s_dat['HRRR_AGL_SPD']+s_dat['HRRR_10M_SPD']+s_dat['GEOSFP_50M_SPD'])/3
fspd = (f_dat['HRRR_AGL_SPD']+f_dat['HRRR_10M_SPD']+f_dat['GEOSFP_50M_SPD'])/3
plt.scatter(s_dat['CAMPD_RATE'], sspd, c='g')
plt.scatter(f_dat['CAMPD_RATE'], fspd, c='r')
plt.xlabel('campd emission rate kg/s')
plt.ylabel('avg wind speed m/s')

In [ ]:
data.loc[data["GRANULE"] == "20241201T171748_2433612_030", "PLUME"] = True
data.loc[data["GRANULE"] == "20240625T160755_2417711_031", "PLUME"] = True

data.to_csv(csv_savefn, index=False)